# HE-MNIST Walkthrough

This notebook walks through every concept and every line of code in the project.

## What is Homomorphic Encryption?

Normal encryption lets you store or transmit data safely, but to *compute* on it you must decrypt first — exposing the plaintext to whoever is running the computation.

**Homomorphic Encryption (HE)** breaks that constraint:

```
Encrypt(a) ⊕ Encrypt(b)  =  Encrypt(a + b)
Encrypt(a) ⊗ Encrypt(b)  =  Encrypt(a × b)
```

The server can add and multiply ciphertexts and get back the *encrypted result* — without ever learning `a` or `b`.

In our case: a hospital encrypts a patient's retinal scan, sends it to a cloud ML service, and gets back an encrypted diagnosis. The cloud sees only noise at every step.

## The CKKS Scheme

There are several HE schemes. We use **CKKS** (Cheon-Kim-Kim-Song, 2017) because:

| Scheme | Supports | Use case |
|--------|----------|----------|
| BFV / BGV | Exact integers | Counting, comparisons |
| **CKKS** | **Approximate real numbers** | **Neural networks, statistics** |

CKKS encodes a float vector into a polynomial ring element. Arithmetic on the polynomial corresponds to approximate arithmetic on the floats. The approximation error is tiny and controllable — think of it like floating-point rounding.

### Key parameters

```
poly_modulus_degree  (n)  — ring dimension, must be power of 2
coeff_mod_bit_sizes       — the modulus "chain"; each middle entry = one operation slot
scale                     — fixed-point precision (2^40 ≈ 40-bit mantissa)
```

### The noise budget — what actually consumes a slot

The critical thing to understand about TenSEAL is that **both** `square` and `mm` (matrix multiply) each consume one chain slot:

- `square` multiplies ciphertext × ciphertext → scale doubles → rescale → slot consumed
- `mm` multiplies ciphertext × plaintext → scale doubles → rescale → slot consumed

Our forward pass has 5 operations: `mm → sq → mm → sq → mm`

That requires depth 5. We provision **6 middle slots** for one slot of headroom:

```
[60,  40, 40, 40, 40, 40, 40,  60]
 ↑                               ↑
special                       special
      |←  6 middle slots  →|
```

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

import numpy as np
import tenseal as ts
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time

print(f"TenSEAL version: {ts.__version__}")
print(f"PyTorch version: {torch.__version__}")

## Part 1 — HE Arithmetic from Scratch

Before touching MNIST, let's build intuition with toy examples.

In [ ]:
# Build a CKKS context
ctx = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=16384,
    coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 40, 40, 60],
)
ctx.global_scale = 2 ** 40
ctx.generate_galois_keys()  # required for vector rotations

print("Context created. Parameters:")
print(f"  poly_modulus_degree : 16384")
print(f"  coeff_mod_bit_sizes : [60, 40×6, 60]  (6 middle slots, depth 6)")
print(f"  total modulus bits  : {60 + 40*6 + 60}  (limit for n=16384 is ~438)")
print(f"  scale               : 2^40 ≈ {2**40:.2e}")
print(f"  forward pass ops    : mm, sq, mm, sq, mm  = 5 slots consumed")

In [ ]:
# Encrypt two vectors and add them
a = [1.0, 2.0, 3.0, 4.0]
b = [10.0, 20.0, 30.0, 40.0]

enc_a = ts.ckks_vector(ctx, a)
enc_b = ts.ckks_vector(ctx, b)

enc_sum = enc_a + enc_b          # addition on ciphertext
result = enc_sum.decrypt()

print("Plaintext a:        ", a)
print("Plaintext b:        ", b)
print("Expected a+b:       ", [x+y for x,y in zip(a,b)])
print("Decrypted (enc_a + enc_b):", [round(x, 6) for x in result])
print("\nNotice the tiny rounding error — that's CKKS's approximate arithmetic.")

In [ ]:
# Multiply ciphertext × ciphertext (costs 1 noise level)
enc_product = enc_a * enc_b
result = enc_product.decrypt()

print("Expected a*b:              ", [x*y for x,y in zip(a,b)])
print("Decrypted (enc_a * enc_b): ", [round(x, 4) for x in result])
print("\nMultiplication works! Now we've consumed 1 of our 2 noise levels.")

In [ ]:
# Multiply ciphertext × plaintext (FREE — does NOT consume a noise level)
enc_scaled = enc_a * 5.0
result = enc_scaled.decrypt()

print("Expected a * 5.0:              ", [x*5 for x in a])
print("Decrypted (enc_a * 5.0):       ", [round(x, 6) for x in result])
print("\nCiphertext × plaintext is free! This is how linear layers work in HE.")
print("Weight matrices are plaintext → matmul costs zero noise levels.")

### Why activations are the hard part

ReLU: `max(0, x)` — not a polynomial. **Cannot be computed on ciphertext.**

We need a polynomial activation. The simplest is **x²** (squaring):
- It's degree 2, so it costs exactly **one multiplication level**
- It's a valid nonlinearity (the network can still learn)
- Alternatives: degree-2/3/4 polynomial approximations of ReLU (more expressive, more depth cost)

In [ ]:
# Visualize square vs ReLU
x = np.linspace(-2, 2, 200)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].plot(x, np.maximum(0, x), 'b', lw=2, label='ReLU (not HE-friendly)')
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_title('ReLU activation')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, x**2, 'r', lw=2, label='x² (HE-friendly polynomial)')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_title('Square activation (x²)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 2 — The HE-Friendly Neural Network

Architecture: **784 → 128 → 64 → 10**

```
Input  →  mm  →  x²  →  mm  →  x²  →  mm  →  Logits
          ↓        ↓     ↓       ↓     ↓
        slot1   slot2  slot3  slot4  slot5
```

Every operation (both `mm` and `square`) consumes one chain slot because each one
multiplies polynomials together and then rescales. Five operations → depth 5 required.
We provision 6 slots to have one in reserve.

In [ ]:
from src.model import HEFriendlyNet

model = HEFriendlyNet()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## Part 3 — Load MNIST and Train

In [ ]:
from src.data import get_dataloaders, get_flat_test_samples

train_loader, test_loader = get_dataloaders(data_dir='../data')

# Show a few samples
images, labels = next(iter(test_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(str(labels[i].item()))
    ax.axis('off')
plt.suptitle('Sample MNIST test images (normalized)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import os, torch
from src.train import train, evaluate

MODEL_PATH = '../model.pt'

model = HEFriendlyNet()
if os.path.exists(MODEL_PATH):
    print('Loading cached model...')
    model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
else:
    print('Training from scratch (this will take ~1-2 min)...')
    # re-point data_dir
    from src import data as _d
    _orig = _d.get_dataloaders.__defaults__
    model = train(epochs=10, save_path=MODEL_PATH)

model.eval()
acc = evaluate(model, test_loader, torch.device('cpu'))
print(f'Plaintext test accuracy: {acc:.2%}')

## Part 4 — Encrypting Images

In [ ]:
from src.encrypt import build_context, encrypt_image, encrypt_batch

he_ctx = build_context()

# Load 5 test images as flat vectors
flat_images, true_labels = get_flat_test_samples(n=5, data_dir='../data')

print(f"Each image: {len(flat_images[0])} float values (28×28 pixels, flattened)")
print(f"Value range: [{min(flat_images[0]):.3f}, {max(flat_images[0]):.3f}] (normalized)")

# Encrypt one image and inspect
t0 = time.time()
enc_img = encrypt_image(he_ctx, flat_images[0])
enc_time = time.time() - t0

print(f"\nEncryption time: {enc_time*1000:.1f} ms")
print(f"Type: {type(enc_img)}")
print(f"True label: {true_labels[0]}")
print("\nThe ciphertext is opaque — we cannot read pixel values from it.")

In [ ]:
# Verify round-trip: encrypt → decrypt gives back the original values
decrypted = enc_img.decrypt()
orig = flat_images[0]

errors = [abs(d - o) for d, o in zip(decrypted, orig)]
print(f"Max round-trip error:  {max(errors):.2e}")
print(f"Mean round-trip error: {np.mean(errors):.2e}")
print("\nThe approximation error is negligible (CKKS uses ~40-bit precision).")

## Part 5 — HE Inference

Now for the main event. We run the neural network forward pass **entirely on ciphertext**.

The model weights are plaintext. Only the image stays encrypted throughout.

```
enc_x  →  [W1 @ enc_x + b1]  →  [enc_x²]  →  [W2 @ enc_x + b2]  →  [enc_x²]  →  [W3 @ enc_x + b3]  →  decrypt  →  argmax
           plaintext matmul       depth=1      plaintext matmul       depth=2      plaintext matmul
```

In [ ]:
from src.he_inference import he_linear, he_square, he_forward

weights = model.get_weights()

print("Running HE forward pass. Slot budget: 6. Each mm and sq costs 1.\n")

t0 = time.time()
x = enc_img

print("Step 1: mm1  — consumes slot 1")
x = he_linear(x, weights['w1'], weights['b1'])

print("Step 2: sq1  — consumes slot 2")
x = he_square(x)

print("Step 3: mm2  — consumes slot 3")
x = he_linear(x, weights['w2'], weights['b2'])

print("Step 4: sq2  — consumes slot 4")
x = he_square(x)

print("Step 5: mm3  — consumes slot 5  (1 slot remaining in reserve)")
x = he_linear(x, weights['w3'], weights['b3'])

print("Step 6: decrypt logits")
logits = x.decrypt()

elapsed = time.time() - t0
pred = int(np.argmax(logits))
print(f"\nTime: {elapsed:.2f}s  |  True: {true_labels[0]}  Pred: {pred}  Correct: {pred == true_labels[0]}")

In [ ]:
# Visualize logits vs plaintext predictions
with torch.no_grad():
    img_tensor = torch.tensor(flat_images[0], dtype=torch.float32).unsqueeze(0)
    plain_logits = model(img_tensor).squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 3))

# Show original image
axes[0].imshow(np.array(flat_images[0]).reshape(28, 28), cmap='gray')
axes[0].set_title(f'Input image (label: {true_labels[0]})')
axes[0].axis('off')

# Plaintext logits
axes[1].bar(range(10), plain_logits, color='steelblue')
axes[1].set_title('Plaintext logits')
axes[1].set_xlabel('Class')
axes[1].set_xticks(range(10))

# HE logits
axes[2].bar(range(10), logits, color='salmon')
axes[2].set_title('HE logits (after decrypt)')
axes[2].set_xlabel('Class')
axes[2].set_xticks(range(10))

plt.tight_layout()
plt.show()

print(f"Plaintext pred: {int(np.argmax(plain_logits))}")
print(f"HE pred:        {int(np.argmax(logits))}")
print(f"Max logit difference: {max(abs(h-p) for h,p in zip(logits, plain_logits)):.4f}")

## Part 6 — Batch Evaluation

In [ ]:
from src.he_inference import run_he_inference

N = 5  # increase to test more (each sample takes ~5-30s)
flat_images_n, labels_n = get_flat_test_samples(n=N, data_dir='../data')

print(f"Encrypting {N} images...")
enc_images_n = encrypt_batch(he_ctx, flat_images_n)

print(f"Running HE inference on {N} encrypted images...\n")
t0 = time.time()
results = run_he_inference(enc_images_n, weights, labels_n)
total = time.time() - t0

print(f"\nTotal time: {total:.1f}s | Per sample: {total/N:.1f}s")

## Part 7 — What We Learned

### Key takeaways

| Concept | What we saw |
|---|---|
| **Ciphertext arithmetic** | Add/multiply directly on encrypted vectors |
| **Noise budget** | Each `*` costs one chain level; linear ops are free |
| **CKKS approximation** | Tiny rounding errors (< 1e-6), negligible for classification |
| **HE-friendly activations** | x² instead of ReLU — same depth cost for both layers |
| **Private inference** | Model weights are public; only the input is private |

### Performance reality check

| Operation | Plaintext | HE |
|---|---|---|
| Single inference | ~0.1 ms | ~5–30 s |
| Overhead | 1× | ~10,000–100,000× |

This is typical for current HE. Active research directions include:
- **Batching** (SIMD packing): encode many images into one ciphertext slot → amortize cost
- **Hardware acceleration**: GPU/FPGA implementations (10–100× speedup)
- **Approximate bootstrapping**: refresh noise budget to support deeper networks
- **Hybrid protocols**: combine HE with Garbled Circuits or Secret Sharing

### Further reading
- [TenSEAL documentation](https://github.com/OpenMined/TenSEAL)
- [CryptoNets paper](https://proceedings.mlr.press/v48/gilad-bachrach16.html) — original HE inference on MNIST
- [SEAL library](https://github.com/microsoft/SEAL) — Microsoft's underlying C++ HE library